In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3" 

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader
from vllm import LLM, SamplingParams

In [2]:
model_path = "/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/pos_rs_0.1-neg_rs_0.01-equ+belu+rule-piecewise_error/actor/global_step_200"

tokenizer = AutoTokenizer.from_pretrained(model_path)
torch.cuda.empty_cache()
base_model = LLM(
    model=model_path,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.85,
    dtype="auto"
)

INFO 08-19 10:39:49 config.py:1450] Downcasting torch.float32 to torch.float16.
INFO 08-19 10:39:49 llm_engine.py:174] Initializing an LLM engine (v0.5.4) with config: model='/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/pos_rs_0.1-neg_rs_0.01-equ+belu+rule-piecewise_error/actor/global_step_200', speculative_config=None, tokenizer='/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/pos_rs_0.1-neg_rs_0.01-equ+belu+rule-piecewise_error/actor/global_step_200', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=16384, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityCo

Loading safetensors checkpoint shards:   0% Completed | 0/7 [00:00<?, ?it/s]


INFO 08-19 10:39:56 model_runner.py:732] Loading model weights took 14.2448 GB
INFO 08-19 10:39:57 gpu_executor.py:102] # GPU blocks: 59353, # CPU blocks: 4681
INFO 08-19 10:40:00 model_runner.py:1024] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 08-19 10:40:00 model_runner.py:1028] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 08-19 10:40:19 model_runner.py:1225] Graph capturing finished in 18 secs.


In [3]:
val_data_path = "dataset/eval.passn.parquet"
val_dataset = RLHFDataset(parquet_files=val_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

val_dataloader = DataLoader(dataset=val_dataset,
                            batch_size=128,
                            shuffle=False,
                            drop_last=False,
                            collate_fn=collate_fn)
n_val_samples = 8

original dataset len: 1590
filter dataset len: 1588


In [4]:
pad_token_id = tokenizer.pad_token_id
from typing import List
def _pre_process_inputs(pad_token_id, prompt_token_ids: torch.Tensor) -> List[int]:
    # remove the left padding in the prompt token_id
    # pad_token_id = self.llm_engine.tokenizer.pad_token_id if self.llm_engine.tokenizer.pad_token_id is not None else self.llm_engine.tokenizer.eos_token_id
    non_pad_index = torch.nonzero(prompt_token_ids != pad_token_id, as_tuple=False)[0][0]
    token_ids = prompt_token_ids[non_pad_index:].tolist()
    return token_ids


In [5]:
from deepscaler.globals import THOUGHT_DELIMITER_START, THOUGHT_DELIMITER_END, OAI_RM_MODEL
from deepscaler.rewards import RewardConfig, RewardFn, RewardInput, RewardOutput, RewardType
from deepscaler.rewards.math_reward import RewardMathFn
from verl.div_src.reward_fn import RewardManager
reward_fn = RewardMathFn(RewardConfig)
input = RewardInput(problem="", problem_type=RewardType.MATH, model_response="<think> I am omniscient. </think> The answer is \\boxed{24 + 14*x + (-13)*x^2 - 2*x^3 + x^4}.", ground_truth={"answer": "$x^{4}-2 x^{3}-13 x^{2}+14 x+24$"})
output = reward_fn(input)
print(output.reward)
# THOUGHT_DELIMITER_START + output[0].outputs[0].text
# reward_fn = RewardManager(tokenizer=tokenizer, num_examine=0)

1.0


In [ ]:
from verl import DataProto
reward_tensor_lst = []
data_source_lst = []
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=1.0,
    max_tokens=8192,
)
val_reward_fn = RewardManager(tokenizer=tokenizer, num_examine=1)
for test_data in val_dataloader:
    
    test_batch = DataProto.from_single_dict(test_data)
    test_batch = test_batch.repeat(repeat_times=n_val_samples, interleave=True)
    idx = test_batch.batch['input_ids']
    batch_size = idx.size(0)

    idx_list = [_pre_process_inputs(pad_token_id, idx[i]) for i in range(batch_size)]
    output = base_model.generate(prompts=None,
                                 sampling_params=sampling_params,
                                 prompt_token_ids=idx_list,
                                 use_tqdm=True)
    data_source_lst.append(test_batch.non_tensor_batch.get('data_source', ['unknown'] * batch_size))
    
    def process_item(response, ground_truth):
        reward_input = RewardInput(problem="", problem_type=RewardType.MATH, model_response=THOUGHT_DELIMITER_START+response, ground_truth={"answer": ground_truth})
        return reward_fn(reward_input).is_correct
    # print(test_batch['reward_model'])
    reward_tensor = torch.zeros(batch_size)
    for i in range(batch_size):
        reward_i = process_item(output[i].outputs[0].text, test_batch.non_tensor_batch['reward_model'][i]['ground_truth'])
        reward_tensor[i] = reward_i
    reward_tensor_lst.append(reward_tensor)
    

Processed prompts:  23%|██▎       | 236/1024 [01:23<01:57,  6.69it/s, est. speed input: 642.62 toks/s, output: 1967.96 toks/s]

In [ ]:
# print(len(reward_tensor_lst[0]))
import numpy as np
reward_tensors = torch.cat(reward_tensor_lst, dim=0).reshape(-1, n_val_samples).any(dim=-1)
data_sources = np.concatenate(data_source_lst, axis=0).reshape(-1, n_val_samples)[:,0]
data_source_reward = {}
for i in range(reward_tensors.shape[0]):
    data_source = data_sources[i]
    if data_source not in data_source_reward:
        data_source_reward[data_source] = []
    data_source_reward[data_source].append(reward_tensors[i])
metric_dict = {}
average_score = []
for data_source, rewards in data_source_reward.items():
    metric_dict[f'val/test_score/{data_source}'] = np.mean(rewards)
    average_score.append(metric_dict[f'val/test_score/{data_source}'])
metric_dict[f'val/test_score/average'] = np.mean(average_score)
print(metric_dict)

In [ ]:
# for i in range(batch_size):
#     print(f"**************{i}**************")
#     print("ground_truth:", test_batch.non_tensor_batch['reward_model'][i]['ground_truth'])
#     print(output[i].outputs[0].text)
#     print("_______________________________")